In [19]:
# Import packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Data Cleaning
## Cleaning file: "PrimeTransactions"

In [31]:
import csv

# Open the file and count lines without loading them into a DataFrame
with open('PrimeTransactions.csv', 'r', encoding='utf-8') as f:
    row_count = sum(1 for row in f) - 1 # Subtract 1 for the header row

print(f"Total rows in the raw CSV: {row_count:,}")

Total rows in the raw CSV: 11,027


In [35]:
#Eliminated the print statement to save space in the notebook
headers = pd.read_csv('PrimeTransactions.csv', nrows=0).columns.tolist()
headers

['contract_transaction_unique_key',
 'contract_award_unique_key',
 'award_id_piid',
 'modification_number',
 'transaction_number',
 'parent_award_agency_id',
 'parent_award_agency_name',
 'parent_award_id_piid',
 'parent_award_modification_number',
 'federal_action_obligation',
 'total_dollars_obligated',
 'total_outlayed_amount_for_overall_award',
 'base_and_exercised_options_value',
 'current_total_value_of_award',
 'base_and_all_options_value',
 'potential_total_value_of_award',
 'disaster_emergency_fund_codes_for_overall_award',
 'outlayed_amount_from_COVID-19_supplementals_for_overall_award',
 'obligated_amount_from_COVID-19_supplementals_for_overall_award',
 'outlayed_amount_from_IIJA_supplemental_for_overall_award',
 'obligated_amount_from_IIJA_supplemental_for_overall_award',
 'action_date',
 'action_date_fiscal_year',
 'period_of_performance_start_date',
 'period_of_performance_current_end_date',
 'period_of_performance_potential_end_date',
 'ordering_period_end_date',
 'solic

### Classifying useful columns
#### Financial indicators
1. federal_action_obligation:Dollar amount the U.S. government legally commits to an award, contract, or grant transaction.
2. total_dollars_obligated: Useful. Shows the cumulative cost.
3. base_and_all_options_value: This value represents the maximum potential spending on a contract if every option is exercised.
4. current_total_value_of_award: Total amount that will be paid out to a recipient if they fulfill all commitments under the currently-exercised options of a contract.

#### Operational Drivers
1. modification_number: Unique identifier issued by federal agencies to track changes, amendments, or revisions to existing contracts, grants, or financial assistance agreements.
2. period_of_performance_start_date & period_of_performance_current_end_date: Useful. Used to calculate Duration.
3. type_of_contract_pricing: Identifies the specific pricing structure of a federal contract, as defined in Federal Acquisition Regulation (FAR) Part 16. 
4. number_of_offers_received: Useful. A proxy for competition. Does more competition lead to lower costs? **Note to self: OLS can answer that.**
5. solicitation_date: Comparing this to the period_of_performance_start_date to calculate Procurement Lead Time.

#### Contextual & Categorical (Useful for Visualization)
1. recipient_name: Useful. Who is the engineer/contractor?
2. awarding_sub_agency_name: Useful. Which branch of the DOE is spending?
3. primary_place_of_performance_state_name: Useful. **Note to self: Essential for creating a geographic map.**
4. naics_description: Useful. Industry.
5. product_or_service_code_description: Additional dimension to group data for further analysis.
6. unique_extent_competed: Usefull variable to test in OLS model

#### Metadata
1. contract_transaction_unique_key & award_id_piid: Not Useful for analysis. These are just ID tags.
2. parent_award_agency_id: Not Useful. Could be useful in a more granular analysis of agency hierarchies.
3. usaspending_permalink: Not Useful. It's just a URL.

#### Other (Likely Not Useful)
Note to self: Many of these columns are specific to certain laws or small subsets of data.
1. disaster_emergency_fund_codes: Not Useful. Unless analyzing projects specifically about COVID-19 or Hurricane relief.
2. highly_compensated_officer_..._amount: Not relevant for Operations Analysis.
3. foreign_funding: Not Useful. Most DOE spending is domestic.
4. recipient_fax_number / phone_number: Not Useful.

Sources:
https://cpars.cpars.gov/cpars/common/helpinfo_input.action?module=cpars&scope=de&topic=Index

In [34]:
useful_cols = [
    # Metadata (For Indexing/Labeling)
    'award_id_piid', 
    
    # Financial Indicators (Y-Variables)
    'federal_action_obligation', 
    'total_dollars_obligated', 
    'base_and_all_options_value',
    'current_total_value_of_award',
    
    # Operational Drivers (X-Variables)
    'modification_number',
    'period_of_performance_start_date',
    'period_of_performance_current_end_date',
    'solicitation_date', # Added for Procurement Lead Time
    'type_of_contract_pricing',
    'number_of_offers_received',
    'extent_competed', # Added for OLS testing
    
    # Contextual & Categorical (For Visualization/Grouping)
    'recipient_name',
    'awarding_sub_agency_name',
    'primary_place_of_performance_state_name',
    'naics_description',
    'product_or_service_code_description'
]

clean_df = prime_transactions[useful_cols].copy()

print(f"Dataset Dimensions: {clean_df.shape}")
clean_df.head(20)

Dataset Dimensions: (11027, 17)


,award_id_piid,federal_action_obligation,total_dollars_obligated,base_and_all_options_value,current_total_value_of_award,modification_number,period_of_performance_start_date,period_of_performance_current_end_date,solicitation_date,type_of_contract_pricing,number_of_offers_received,extent_competed,recipient_name,awarding_sub_agency_name,primary_place_of_performance_state_name,naics_description,product_or_service_code_description
0,89243323PFE000652,0.00,1.600000e+04,0.00,1.600000e+04,P00001,2023-06-22,2023-07-22,NaN,FIRM FIXED PRICE,1.0,NOT COMPETED,"MATERIALS DESIGN, INC.",Department of Energy,WEST VIRGINIA,OTHER COMPUTER RELATED SERVICES,IT AND TELECOM - BUSINESS APPLICATION SOFTWARE...
1,89243324PFE000700,89518.00,8.951800e+04,89518.00,8.951800e+04,0,2023-12-15,2024-12-31,2023-10-24,FIRM FIXED PRICE,1.0,NOT COMPETED,DRS ARCHITECTS INC,Department of Energy,OREGON,ENGINEERING SERVICES,ARCHITECT AND ENGINEERING- GENERAL: OTHER
2,89303024PAU000047,31750.00,3.175000e+04,31750.00,3.175000e+04,0,2024-08-29,2025-02-28,NaN,FIRM FIXED PRICE,1.0,NOT COMPETED UNDER SAP,"AMBITEC, INC.",Department of Energy,DISTRICT OF COLUMBIA,OTHER PROFESSIONAL EQUIPMENT AND SUPPLIES MERC...,SAFETY AND RESCUE EQUIPMENT
3,DEAC0206CH11357,16431217.58,1.504272e+10,18677709.67,1.558112e+10,1001,2006-07-31,2026-09-30,NaN,COST PLUS AWARD FEE,1.0,FULL AND OPEN COMPETITION,"UCHICAGO ARGONNE, LLC",Department of Energy,ILLINOIS,"RESEARCH AND DEVELOPMENT IN THE PHYSICAL, ENGI...",OPER OF GOVT R&D GOCO FACILITIES
4,89233121PNA000132,60000.00,1.186836e+05,34800.00,1.186836e+05,P00005,2021-09-20,2024-09-19,NaN,FIRM FIXED PRICE,3.0,COMPETED UNDER SAP,"ATLANTIC DIVING SUPPLY, INC.",Department of Energy,VIRGINIA,"SEARCH, DETECTION, NAVIGATION, GUIDANCE, AERON...","NIGHT VISION EQUIPMENT, EMITTED AND REFLECTED ..."
5,89243118CSC000013,5000.00,5.489596e+07,500.00,6.676460e+07,P00105,2018-07-23,2024-04-30,NaN,COST PLUS AWARD FEE,4.0,FULL AND OPEN COMPETITION AFTER EXCLUSION OF S...,"GOLDEN SVCS, LLC",Department of Energy,TENNESSEE,SECURITY GUARDS AND PATROL SERVICES,SUPPORT- PROFESSIONAL: PHYSICAL SECURITY AND B...
6,89303021CHC000012,56250.00,3.140572e+06,-39995.08,4.000000e+06,P00017,2021-08-02,2024-08-01,2021-06-10,LABOR HOURS,1.0,NOT AVAILABLE FOR COMPETITION,"ALEUT TECHNICAL SERVICES, LLC",Department of Energy,DISTRICT OF COLUMBIA,"ALL OTHER PROFESSIONAL, SCIENTIFIC, AND TECHNI...",SUPPORT- ADMINISTRATIVE: OTHER
7,89303023CMA000093,200000.00,8.080000e+05,1654.84,8.080000e+05,P00005,2023-10-01,2024-09-30,2023-05-01,LABOR HOURS,1.0,NOT AVAILABLE FOR COMPETITION,ATI GOVERNMENT SOLUTIONS LLC,Department of Energy,DISTRICT OF COLUMBIA,ADMINISTRATIVE MANAGEMENT AND GENERAL MANAGEME...,IT AND TELECOM - DATA CENTER PRODUCTS (HARDWAR...
8,89243323CFE000075,3450642.00,9.954369e+07,0.00,1.169775e+08,P00054,2023-02-01,2026-01-31,2021-09-17,COST PLUS AWARD FEE,2.0,FULL AND OPEN COMPETITION,"KEYLOGIC, LLC",Department of Energy,WEST VIRGINIA,ENGINEERING SERVICES,SUPPORT- PROFESSIONAL: ENGINEERING/TECHNICAL
9,89303122PEM000013,0.00,1.446106e+04,0.00,1.446106e+04,P00001,2022-06-06,2023-06-05,NaN,FIRM FIXED PRICE,1.0,NOT COMPETED,DELTEK INC.,Department of Energy,KENTUCKY,OTHER COMPUTER RELATED SERVICES,IT AND TELECOM - APPLICATION DEVELOPMENT SOFTW...


## Cleaning file "Subawards"

In [32]:
import csv

# Open the file and count lines without loading them into a DataFrame
with open('Subawards.csv', 'r', encoding='utf-8') as f:
    row_count = sum(1 for row in f) - 1 # Subtract 1 for the header row

print(f"Total rows in the raw CSV: {row_count:,}")

Total rows in the raw CSV: 10,701


In [36]:
#Eliminated the print statement to save space in the notebook
headers = pd.read_csv('Subawards.csv', nrows=0).columns.tolist()
#headers

### Classifying useful columns
#### Link Keys
1. prime_award_piid: To match the award_id_piid in the Prime_Transactions file.
2. prime_award_unique_key: Secondary backup key for merging.

#### Subaward Financials
1. subaward_amount: The amount given to the sub-contractor.
2. prime_award_amount: Essential to calculate what percentage of the total project is being outsourced.
3. subaward_action_date: When the sub-contract was issued.

##### Subaward Operations & Context

1. prime_award_base_action_date_fiscal_year:
2. prime_award_latest_action_date: Useful to see the "age" of the prime contract at the moment the subaward was issued. **PENDING**
3. prime_award_period_of_performance_start_date: Test to know the "arrival" and "departure" of a subcontractor
4. prime_award_period_of_performance_current_end_date: Defines the current funded "deadline" for the work.
5. prime_award_period_of_performance_potential_end_date: Defines the total "Runway" or planned lifecycle of the engineering project.

#### Prime Agency Info

1. prime_award_awarding_agency_name: Keep. Confirms this is Department of Energy.
2. prime_award_awarding_sub_agency_name: Keep. Essential for grouping (e.g., Office of Science vs. NNSA).

#### Awardee Info

1. prime_awardee_name: Keep. Identifies the main contractor.
2. prime_awardee_city_name: Keep. Useful for city-level spending trends.
3. prime_awardee_state_name: Keep. Crucial for Tableau mapping.
4. prime_awardee_business_types: Keep. Tells you the "Risk Profile" of the prime (e.g., Non-profit, Small Business).

#### Project details

1. prime_award_project_title: Keep. Provides the "Human-readable" name of the project.
2. prime_award_naics_description: Keep. Identifies the industry of the Prime.

#### Subaward Details

1. subaward_number: Keep. To track specific tasks within a PIID.
2. subaward_amount: Keep. The "Money" variable for your sub-analysis.
3. subaward_action_date: Keep. The "Arrival" date of the specialist.
4. subawardee_name: Keep. Identifies who is actually doing the outsourced work.
5. subawardee_state_name: Keep. Shows the geographical "reach" of the subcontracting.
6. subawardee_business_types: Keep. Vital for analyzing if subcontracts are going to small businesses.
7. subaward_description: Keep. Often provides the most technical engineering detail.